# PoC v2 — Extração Estruturada com Schema Granular

**Evolução da v1.** Este notebook segue o fluxo operacional abaixo:

`AmostraDescMerc.csv` (o CSV de entrada já usado anteriormente)
→ coluna `Text` / `XPROD`
→ **Attack Augmentation**
→ `attack.csv` (`original`, `attacked`) gerado com ataque em palavras inteiras
→ **Normalization**
→ `normalized.csv` (`original`, `attacked`, `normalized`)
→ **Benchmark**
→ `resultado`

A normalização mantém a constância esperada do produto no formato:

`${nome} ${marca} ${cor} {...} - ${quantidade} ${unidade}`

> Toda decisão é explicável, mensurável e replicável, com a entrada atacada separada da referência original para avaliação.

# PoC v2 — Extração Estruturada com Schema Granular

**Evolução da v1.** O problema central da v1 é que `Nome_Normalizado` engloba tudo num único campo de texto livre. Isso causa dois problemas operacionais sérios:

1. **Omissão de variantes**: `Desodorante Rexona Erva Doce 150ml` e `Desodorante Rexona Invisible 150ml` geram nomes diferentes, impossibilitando agrupamento confiável.
2. **Impossibilidade de agrupamento por produto canônico**: sem separação de atributos, qualquer comparação depende de fuzzy matching frágil.

### Novo schema — 5 campos separados

| Campo | Descrição |
|---|---|
| `Categoria` | Substantivo genérico do produto (ex: Desodorante, Shampoo) |
| `Marca` | Marca comercial, mesmo que abreviada |
| `Tipo_Variante` | Fragrância, linha ou fórmula (ex: Jasmim, Óleo Extraordinário) |
| `Quantidade` | Valor numérico do produto unitário |
| `Unidade_Medida` | Unidade SEFAZ (Mililitros, Gramas, Litros, etc.) |

**Produto canônico para agrupamento** = `Categoria + Marca + Tipo_Variante`.  
Dois itens com o mesmo trio são o mesmo produto, independente de como foram escritos na nota fiscal.


## 2. Dicionário de Canonicalização de Marcas

A canonicalização de marcas é feita **pós-inferência**, via lookup semântico — não por regex.

Isso é filosoficamente diferente da abordagem de extração por padrão fixo: o LLM extrai a marca pelo contexto semântico (mesmo abreviada), e o dicionário apenas padroniza o resultado para uma forma canônica. São duas etapas independentes e auditáveis.


In [1]:
# DICIONÁRIO DE CANONICALIZAÇÃO DE MARCAS (lookup pós-LLM)

BRAND_ALIASES = {
    "REXO": "Rexona", "REX": "Rexona", "REXONA": "Rexona",
    "DOVE": "Dove", "DV": "Dove",
    "ELSEVE": "Elseve", "ELSEVY": "Elseve",
    "BOZZANO": "Bozzano", "BOZZ": "Bozzano",
    "PANTENE": "Pantene", "PANT": "Pantene",
    "NIVEA": "Nivea", "NIV": "Nivea",
    "GILLETTE": "Gillette", "GIL": "Gillette",
    "ARIEL": "Ariel",
    "OMO": "Omo",
    "FRUCTIS": "Fructis",
    "TRESEMME": "Tresemmé", "TRESEM": "Tresemmé",
    "SEDA": "Seda",
    "CLEAR": "Clear",
    "HEAD": "Head & Shoulders", "H&S": "Head & Shoulders",
    "GARNIER": "Garnier",
    "LOREAL": "L'Oréal", "LOREAL PARIS": "L'Oréal",
    "NATURA": "Natura",
    "AVON": "Avon",
    "JOHNSON": "Johnson & Johnson", "J&J": "Johnson & Johnson",
    "BABY SOFT": "Baby Soft", "BABYSOFT": "Baby Soft",
    "DOWNY": "Downy",
    "COMFORT": "Comfort",
    "ACE": "Ace",
    "VANISH": "Vanish",
    "VEJA": "Veja",
    "AJAX": "Ajax",
    "BRILHANTE": "Brilhante",
    "SURF": "Surf",
}

def canonicalize_brand(raw: str) -> str:
    """Lookup semântico pós-inferência — não é regex, é resolução de alias."""
    if not raw or not isinstance(raw, str):
        return raw
    key = raw.upper().strip()
    return BRAND_ALIASES.get(key, raw.strip().title())

# Teste rápido
assert canonicalize_brand("REXO") == "Rexona"
assert canonicalize_brand("TRESEM") == "Tresemmé"
assert canonicalize_brand("Baby Soft") == "Baby Soft"
print("Dicionário de canonicalização carregado:", len(BRAND_ALIASES), "aliases")


Dicionário de canonicalização carregado: 41 aliases


## 3. Novo System Prompt — Schema de 5 Campos

O prompt instrui o LLM a extrair cada atributo separadamente, com few-shots que cobrem os casos mais ruidosos do domínio fiscal brasileiro.

**Regras críticas embutidas no prompt:**
- `Tipo_Variante` captura fragrância, linha e variante de fórmula — nunca omitir se presente
- `Marca` deve ser extraída mesmo abreviada (o dicionário corrige depois)
- NUNCA multiplicar fardos — `10X400ML` → Quantidade = 400
- `Categoria` é o substantivo genérico, nunca a marca


### 3.1 Text attack na entrada (XPROD)

Cada descrição de item enviada ao LLM passa primeiro por **`apply_text_attack`**: perturbações **nível palavra inteira** no próprio notebook (sem alterar os caracteres internos das palavras; aqui sem dependência externa, para o fluxo ficar autocontido).

Objetivo: simular ruído extra em **palavras inteiras** (troca, omissão ou repetição de termos) **só no texto que o modelo vê**; validação e métricas continuam usando o lote original passado ao motor.

Parâmetros do construtor: `text_attack_enabled`, `text_attack_seed`, `text_attack_intensity`.

In [15]:
import json
import random
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def apply_text_attack(text: str, rng: random.Random, *, intensity: float = 0.12) -> str:
    """Perturbações nível palavra inteira no XPROD, sem mexer nos caracteres internos."""
    if not text or not isinstance(text, str):
        return text
    tokens = text.split()
    n = len(tokens)
    if n < 2:
        return text
    ops = max(1, int(n * float(intensity) * rng.random()))
    for _ in range(ops):
        t = rng.random()
        if t < 0.45 and n >= 2:
            i = rng.randrange(n - 1)
            tokens[i], tokens[i + 1] = tokens[i + 1], tokens[i]
        elif t < 0.75 and n >= 2:
            i = rng.randrange(n)
            del tokens[i]
            n = len(tokens)
        else:
            i = rng.randrange(n)
            tokens.insert(i, tokens[i])
            n = len(tokens)
    return " ".join(tokens)


class FiscalLLMEngineV2:
    """Motor de extração estruturada com schema granular de 6 campos e text attack na entrada."""

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        *,
        text_attack_enabled: bool = True,
        text_attack_seed: int = 42,
        text_attack_intensity: float = 0.12,
    ):
        self.client = OpenAI()
        self.model_name = model_name
        self.text_attack_enabled = text_attack_enabled
        self.text_attack_intensity = text_attack_intensity
        self.text_attack_seed = text_attack_seed
        self._attack_rng = random.Random(text_attack_seed)
        print(
            "Text attack config:",
            {
                "enabled": self.text_attack_enabled,
                "seed": text_attack_seed,
                "intensity": self.text_attack_intensity,
            },
        )

    def _build_system_prompt(self) -> str:
        return """
[SISTEMA]
Você é um algoritmo de normalização de descrições de produtos.
Recebe títulos que podem conter abreviações, erros de digitação, notações
de unidade variadas ou estar em português/espanhol.
Sua única função: retornar JSON estruturado. Sem comentários, sem texto extra.

════════════════════════════════════════
CAMPOS A EXTRAIR (obrigatórios)
════════════════════════════════════════
1. Categoria          — tipo genérico do produto em português. NUNCA use a marca.
                        Expanda abreviações: "Microsc." → "Microscópio", "Liq." → "Liquidificador"
2. Marca              — nome da marca comercial. Se ausente, use null. Não invente marcas.
3. Tipo_Variante      — modelo, especificação, cor, variante relevante. Title Case. null se ausente.
4. Quantidade_Unitaria — valor numérico da medida do produto individual.
                         CONVERTA SEMPRE para a unidade base:
                         "0,75L" → 750 (Mililitros) | "1,5 litros" → 1500 (Mililitros)
                         "meio litro" → 500 (Mililitros) | "12 kilos" → 12 (Quilogramas)
                         Use 1 se não houver medida identificável.
5. Quantidade_Comercial — unidades no pacote/kit. "Pack X 13" → 13. 1 se não houver.
6. Unidade_Medida     — unidade de Quantidade_Unitaria. Use exatamente:
                         Mililitros | Litros | Gramas | Quilogramas | Metros | Centímetros | Unidade

════════════════════════════════════════
REGRAS CRÍTICAS
════════════════════════════════════════
- Abreviações → forma canônica em português correto.
  "Microsc. Biol. Binoc." → Categoria="Microscópio", Tipo_Variante="Biológico Binocular"
  "Llav. Cont." → Categoria="Chave de Contato"
  "Bat." → Categoria="Bateria"
- Erros de digitação → corrija silenciosamente (ex: "Samsugn" → Marca="Samsung").
- Unidades variadas → mesmo valor canônico:
  "750ml", "0,75L", "750 mililitros" → Quantidade_Unitaria=750, Unidade_Medida="Mililitros"
- Prefixos de quantidade comercial (ex: "Pack X 13", "11 Packs", "C/100") →
  vão para Quantidade_Comercial, não alteram a Categoria.
- Ignore sufixos irrelevantes: "0km", "Sedan", códigos internos, referências de sistema.
- Tipo_Variante captura especificações técnicas relevantes:
  modelo, cor, material, voltagem, dimensão, sabor, fragrância.

════════════════════════════════════════
OUTPUT SCHEMA
════════════════════════════════════════
{
  "produtos_normalizados": [
    {
      "Categoria": "string",
      "Marca": "string | null",
      "Tipo_Variante": "string | null",
      "Quantidade_Unitaria": number,
      "Quantidade_Comercial": number,
      "Unidade_Medida": "string"
    }
  ]
}

════════════════════════════════════════
REGRA DE ESTABILIDADE DA CATEGORIA
════════════════════════════════════════
A Categoria deve ser SEMPRE o mesmo termo para o mesmo tipo de produto,
independente de como o título foi escrito ou abreviado.

- Use o substantivo mais genérico e estável em português.
- Se o produto for desconhecido ou técnico, use o substantivo principal
  do título original sem traduzir.
  Ex: "Bocallave Hexagonal" → Categoria="Bocallave"
  Ex: "Disyuntor Diferencial" → Categoria="Disjuntor Diferencial"
- NUNCA use sinônimos alternados: escolha UM termo e mantenha.
  "Automóvel" = "Carro" = "Veículo" → use sempre "Automóvel"
  "Switch" = "Comutador" → use sempre "Switch"
  "Microscópio" nunca vira "Equipamento Óptico"
- Abreviações de categoria → sempre expanda para a mesma forma canônica.
  "Microsc." → sempre "Microscópio", nunca "Microscópio Óptico"

════════════════════════════════════════
FEW-SHOT EXAMPLES
════════════════════════════════════════
Entrada: "Microsc. Biol. Binoc. Meopta Prof."
Saída: {"Categoria":"Microscópio","Marca":"Meopta","Tipo_Variante":"Biológico Binocular Profissional","Quantidade_Unitaria":1,"Quantidade_Comercial":1,"Unidade_Medida":"Unidade"}

Entrada: "Vinho Flor De Crasto Douro Doc (0,75L)"
Saída: {"Categoria":"Vinho","Marca":"Flor De Crasto","Tipo_Variante":"Douro Doc","Quantidade_Unitaria":750,"Quantidade_Comercial":1,"Unidade_Medida":"Mililitros"}

Entrada: "Bat. Cr2032 C/100pçs"
Saída: {"Categoria":"Bateria","Marca":null,"Tipo_Variante":"CR2032","Quantidade_Unitaria":1,"Quantidade_Comercial":100,"Unidade_Medida":"Unidade"}

Entrada: "11 Packs Husares A4 Cuadriculadas 5 Perforaciones"
Saída: {"Categoria":"Caderno","Marca":"Husares","Tipo_Variante":"A4 Quadriculado 5 Furos","Quantidade_Unitaria":1,"Quantidade_Comercial":11,"Unidade_Medida":"Unidade"}

Entrada: "Favo De Mel Florada Silvestre 200 Gramas"
Saída: {"Categoria":"Mel","Marca":null,"Tipo_Variante":"Florada Silvestre","Quantidade_Unitaria":200,"Quantidade_Comercial":1,"Unidade_Medida":"Gramas"}

Entrada: "Garrafa Antiga Minuano Limão 0,9 L - Âmbar - Bm"
Saída: {"Categoria":"Garrafa","Marca":"Minuano","Tipo_Variante":"Limão Âmbar","Quantidade_Unitaria":900,"Quantidade_Comercial":1,"Unidade_Medida":"Mililitros"}

Entrada: "SH ELSEVE OLEO EXTRAORD NUTRICAO 1X400ML RH1325600"
Saída: {"Categoria":"Shampoo","Marca":"Elseve","Tipo_Variante":"Óleo Extraordinário Nutrição","Quantidade_Unitaria":400,"Quantidade_Comercial":1,"Unidade_Medida":"Mililitros"}

Entrada: "DOVE SAB BRAN 8X6X90G"
Saída: {"Categoria":"Sabonete","Marca":"Dove","Tipo_Variante":"Branco","Quantidade_Unitaria":90,"Quantidade_Comercial":48,"Unidade_Medida":"Gramas"}
"""


    @staticmethod
    def _build_user_prompt(batch: list) -> str:
        prompt = "[USER]\nAnalise o lote abaixo e retorne o JSON correspondente:\n\n"
        for i, item in enumerate(batch, 1):
            prompt += f"Item {i}: {item}\n"
        return prompt

    def process_batch(self, batch: list) -> dict:
        if not batch:
            return {"produtos_normalizados": []}
        model_batch = [str(x) for x in batch]
        if self.text_attack_enabled:
            model_batch = [
                apply_text_attack(x, self._attack_rng, intensity=self.text_attack_intensity)
                for x in model_batch
            ]
        print(f"Enviando lote de {len(model_batch)} itens para {self.model_name}...")
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self._build_system_prompt()},
                {"role": "user", "content": self._build_user_prompt(model_batch)},
            ],
            temperature=0,
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)

print("FiscalLLMEngineV2 (prompt v4 — 6 campos) + text attack na entrada definidos.")

FiscalLLMEngineV2 (prompt v4 — 6 campos) + text attack na entrada definidos.


## 4. Pós-processamento com Canonicalização

Após a inferência, duas operações determinísticas:

1. **`canonicalize_brand`** — resolve aliases de marca via lookup
2. **`Produto_Canonico`** — chave de agrupamento com todos os 5 atributos:

```
Categoria | Marca | Tipo_Variante | Quantidade | Unidade_Medida
```

Dois XPRODs com o mesmo `Produto_Canonico` são o mesmo SKU, independente de como foram escritos na nota fiscal.

**Exemplos esperados:**
```
"SH ELSEVE OLEO EXTRAORD NUTRICAO 1X400ML RH1325600"
"SHAMPOO ELSEVE OL EXTRAORD 400ML"
→ ambos: Shampoo | Elseve | Óleo Extraordinário Nutrição | 400 | Mililitros

"AMAC BABY SOFT T. CARINHO AZUL 2 LT CX/0006"
"AMACIANTE BABY SOFT CARINHO AZUL 2L"
→ ambos: Amaciante | Baby Soft | Carinho Azul | 2 | Litros
```

In [16]:
# Mapa de conversão para unidade base canônica
_TO_ML = {"Litros": 1000, "Mililitros": 1}
_TO_G  = {"Quilogramas": 1000, "Gramas": 1}

def _normalizar_quantidade(qtd, uom: str):
    """Converte tudo para unidade base: Litros→Mililitros, Quilogramas→Gramas."""
    try:
        v = float(str(qtd).replace(",", "."))
    except (TypeError, ValueError):
        return 1, uom
    if uom == "Litros":
        return int(v * 1000) if (v * 1000) == int(v * 1000) else round(v * 1000, 3), "Mililitros"
    if uom == "Quilogramas":
        return int(v * 1000) if (v * 1000) == int(v * 1000) else round(v * 1000, 3), "Gramas"
    return (int(v) if v == int(v) else v), uom


def post_process(produto: dict) -> dict:
    """Canonicaliza marca, normaliza unidade para base e gera Produto_Canonico."""
    produto = produto.copy()

    # 1. Canonicaliza marca via lookup
    produto["Marca"] = canonicalize_brand(produto.get("Marca") or "")

    # 2. Normaliza quantidade para unidade base (Litros→ml, kg→g)
    qtd_raw = produto.get("Quantidade_Unitaria")
    uom_raw = (produto.get("Unidade_Medida") or "").strip()
    qtd_norm, uom_norm = _normalizar_quantidade(qtd_raw, uom_raw)
    produto["Quantidade_Unitaria"] = qtd_norm
    produto["Unidade_Medida"] = uom_norm

    qtd_str = str(int(qtd_norm)) if qtd_norm == int(qtd_norm) else str(qtd_norm)

    # 3. Produto_Canonico — chave de agrupamento por SKU
    partes = [
        (produto.get("Categoria") or "").strip().title(),
        (produto.get("Marca") or "").strip(),
        (produto.get("Tipo_Variante") or "").strip(),
        qtd_str,
        uom_norm,
    ]
    produto["Produto_Canonico"] = " | ".join(p for p in partes if p).strip(" |")
    return produto

# Testes de sanidade
ex1 = post_process({"Categoria":"Shampoo","Marca":"ELSEVE","Tipo_Variante":"Óleo Extraordinário Nutrição","Quantidade_Unitaria":400,"Quantidade_Comercial":1,"Unidade_Medida":"Mililitros"})
ex2 = post_process({"Categoria":"Sabonete","Marca":"Dove","Tipo_Variante":"Branco","Quantidade_Unitaria":90,"Quantidade_Comercial":48,"Unidade_Medida":"Gramas"})
ex3 = post_process({"Categoria":"Vinho","Marca":"Flor De Crasto","Tipo_Variante":"Douro Doc","Quantidade_Unitaria":0.75,"Quantidade_Comercial":1,"Unidade_Medida":"Litros"})

print("Ex1:", ex1["Produto_Canonico"])
print("Ex2:", ex2["Produto_Canonico"])
print("Ex3 (0.75L → ml):", ex3["Produto_Canonico"])  # deve virar 750 Mililitros


Ex1: Shampoo | Elseve | Óleo Extraordinário Nutrição | 400 | Mililitros
Ex2: Sabonete | Dove | Branco | 90 | Gramas
Ex3 (0.75L → ml): Vinho | Flor De Crasto | Douro Doc | 750 | Mililitros


## 5. Validação do Novo Schema

O `OutputValidator` da v1 checava 3 campos. A v2 checa os 5 campos do novo schema.


In [17]:
class OutputValidatorV2:
    """Validador para o schema granular de 6 campos."""

    CAMPOS_OBRIGATORIOS = {
        "Categoria", "Marca", "Tipo_Variante",
        "Quantidade_Unitaria", "Quantidade_Comercial", "Unidade_Medida"
    }

    @staticmethod
    def validate(lote_input: list, json_output: dict) -> tuple:
        if "produtos_normalizados" not in json_output:
            return False, "Falta a chave raiz 'produtos_normalizados'."
        produtos = json_output["produtos_normalizados"]
        if not isinstance(produtos, list):
            return False, "'produtos_normalizados' deve ser uma lista."
        if len(produtos) != len(lote_input):
            return False, f"Completude falhou: enviado {len(lote_input)}, recebido {len(produtos)}."
        for i, prod in enumerate(produtos, 1):
            if not isinstance(prod, dict):
                return False, f"Item {i} não é objeto JSON."
            faltantes = OutputValidatorV2.CAMPOS_OBRIGATORIOS - prod.keys()
            if faltantes:
                return False, f"Item {i}: campos ausentes {faltantes}."
        return True, "Validação aprovada."


class ResilientFiscalLLMEngineV2(FiscalLLMEngineV2):
    """Extensão resiliente com validação V2 e retentativas."""

    def process_batch_with_retry(self, batch: list, max_retries: int = 3, backoff_seconds: float = 2.0) -> dict:
        for tentativa in range(1, max_retries + 1):
            print(f"Tentativa {tentativa}/{max_retries}...")
            try:
                resultado = self.process_batch(batch)
                is_valid, msg = OutputValidatorV2.validate(batch, resultado)
                if is_valid:
                    print("Validação aprovada.")
                    return resultado
                print(f"Rejeitado: {msg}")
            except Exception as e:
                print(f"Erro: {e}")
            if tentativa < max_retries:
                time.sleep(backoff_seconds * tentativa)
        print("Falha crítica: limite de retentativas atingido.")
        return {"produtos_normalizados": [], "status": "failed"}

print("OutputValidatorV2 (6 campos) e ResilientFiscalLLMEngineV2 definidos.")

OutputValidatorV2 (6 campos) e ResilientFiscalLLMEngineV2 definidos.


## 6. Teste com Casos Reais — v1 vs v2 Side-by-Side

Rodamos os mesmos 3 casos da v1 (amaciante, iogurte, shampoo) mais os casos de desodorante que motivaram o novo schema.


In [18]:
import pandas as pd
import re
from rank_bm25 import BM25Okapi
from typing import List
from pathlib import Path

class TextNormalizer:
    @staticmethod
    def tokenize(text: str) -> List[str]:
        if not isinstance(text, str):
            return []
        text = text.lower().strip()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.split()

class SemanticBlocker:
    def __init__(self, data: pd.DataFrame, text_column: str):
        self.data = data.copy()
        self.text_column = text_column
        self.corpus_tokens = []
        self.bm25_engine = None

    def fit(self):
        textos = self.data[self.text_column].fillna('').astype(str).tolist()
        self.corpus_tokens = [TextNormalizer.tokenize(doc) for doc in textos]
        self.bm25_engine = BM25Okapi(self.corpus_tokens)
        print("Motor BM25 pronto.")

    def get_batch_for(self, query_text: str, batch_size: int = 15) -> List[str]:
        tokenized_query = TextNormalizer.tokenize(query_text)
        return self.bm25_engine.get_top_n(
            query=tokenized_query,
            documents=self.data[self.text_column].fillna('').astype(str).tolist(),
            n=batch_size
        )

DATA_PATH = Path('../data/AmostraDescMerc.csv')
df_sefaz = pd.read_csv(DATA_PATH, sep='\t')
print(f"Base carregada: {len(df_sefaz):,} linhas")

blocker = SemanticBlocker(data=df_sefaz, text_column='Text')
blocker.fit()


Base carregada: 15,000 linhas
Motor BM25 pronto.


In [6]:
# Helper de inferência em lotes (usado nos benchmarks)
import time


def inferir_em_lotes_v2(engine, itens: list, tamanho_lote: int = 6, pausa: float = 1.0) -> list:
    """Executa a inferência em lotes e retorna lista flat de produtos_normalizados."""
    if not itens:
        return []

    predicoes = []
    total = len(itens)

    for ini in range(0, total, tamanho_lote):
        fim = min(ini + tamanho_lote, total)
        lote = itens[ini:fim]
        idx_lote = (ini // tamanho_lote) + 1
        print(f"Lote {idx_lote} (itens {ini + 1}–{fim})...")

        resultado = engine.process_batch_with_retry(lote)
        produtos = resultado.get("produtos_normalizados", [])
        predicoes.extend(produtos)

        if fim < total and pausa > 0:
            time.sleep(pausa)

    return predicoes


In [7]:
motor_v2 = ResilientFiscalLLMEngineV2(
    model_name="gpt-4o-mini",
    text_attack_enabled=True,
    text_attack_seed=42,
    text_attack_intensity=0.25,
)

casos_teste = {
    "A. Amaciante (abreviação)": "AMACIANTE BABY SOFT 2 LITROS",
    "B. Iogurte (ruído extremo)": "IOGURTE LIQUIDO MORANGO VIGOR",
    "C. Shampoo (variante léxica)": "SHAMPOO ELSEVE OLEO EXTRAORDINARIO NUTRICAO 400ML",
    "D. Desodorante sem variante": "DES 400ml REXO",
    "E. Desodorante com variante": "ANTI TRANS DES REX JASMIN 400ml",
}

resultados_v2 = {}

for titulo, query in casos_teste.items():
    print(f"\n{'='*70}")
    print(f"{titulo} | Query: {query}")
    print('='*70)
    lote = blocker.get_batch_for(query_text=query, batch_size=15)
    resultado = motor_v2.process_batch_with_retry(lote)
    produtos = [post_process(p) for p in resultado.get("produtos_normalizados", [])]
    resultados_v2[titulo] = {"lote": lote, "produtos": produtos}

    df_out = pd.DataFrame([
        {"XPROD": orig, "Categoria": p.get("Categoria"), "Marca": p.get("Marca"),
         "Tipo_Variante": p.get("Tipo_Variante"), "Qtd": p.get("Quantidade"),
         "UoM": p.get("Unidade_Medida"), "Produto_Canonico": p.get("Produto_Canonico")}
        for orig, p in zip(lote, produtos)
    ])
    display(df_out)


Text attack config: {'enabled': True, 'seed': 42, 'intensity': 0.25}

A. Amaciante (abreviação) | Query: AMACIANTE BABY SOFT 2 LITROS
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.


,XPROD,Categoria,Marca,Tipo_Variante,Qtd,UoM,Produto_Canonico
0,EQUILIBRIO AMACIANTE 02X05 LITROS,Amaciante,Equilíbrio,NaN,None,Mililitros,Amaciante | Equilíbrio | 2000 | Mililitros
1,EQUILIBRIO COTTON SOFT 06X02 LITROS,Amaciante,Cotton,Equilíbrio Soft,None,Mililitros,Amaciante | Cotton | Equilíbrio Soft | 2000 | ...
2,AMACIANTE 5 LITROS 4X5L MACIEZ,Amaciante,,NaN,None,Mililitros,Amaciante | 5000 | Mililitros
3,AMAC BABY SOFT 5LI AZUL,Amaciante,Baby,Soft Azul,None,Mililitros,Amaciante | Baby | Soft Azul | 5000 | Mililitros
4,AMAC BABY SOFT T. CUIDADO BRANCO 2 LT CX/0006,Amaciante,Baby,Soft T. Cuidado Branco,None,Mililitros,Amaciante | Baby | Soft T. Cuidado Branco | 20...
5,AMAC BABY SOFT T. CARINHO AZUL 2 LT CX/0006,Amaciante,Baby,Soft T. Carinho Azul,None,Mililitros,Amaciante | Baby | Soft T. Carinho Azul | 2000...
6,AMACIANTE NANO SOFT CARINHO 6X2L,Amaciante,,Nano Carinho Soft,None,Mililitros,Amaciante | Nano Carinho Soft | 2000 | Mililitros
7,AMACIANTE NANO SOFT AMOR 6X2L,Amaciante,,Nano Amor,None,Mililitros,Amaciante | Nano Amor | 2000 | Mililitros
8,AMAC BABY SOFT 5LI FLOWERS ROSA,Amaciante,Baby,Soft Flowers Rosa,None,Mililitros,Amaciante | Baby | Soft Flowers Rosa | 5000 | ...
9,AMACIANTE NANO SOFT ENCANTO 6X2L AZUL,Amaciante,,Nano Soft Encanto Azul,None,Mililitros,Amaciante | Nano Soft Encanto Azul | 2000 | Mi...



B. Iogurte (ruído extremo) | Query: IOGURTE LIQUIDO MORANGO VIGOR
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.


,XPROD,Categoria,Marca,Tipo_Variante,Qtd,UoM,Produto_Canonico
0,DANONINHO CRUSH MORANGO IOGURTE 400GX6,Iogurte,Danoninho,Morango Crush,None,Gramas,Iogurte | Danoninho | Morango Crush | 400 | Gr...
1,TORTUGUITA IOGURTE MORANGO DI 408G,Iogurte,Tortuguita,Morango,None,Gramas,Iogurte | Tortuguita | Morango | 408 | Gramas
2,IOGURTE GREGO MORANGO BATAVO 100G,Iogurte,Batavo,Grego Morango,None,Gramas,Iogurte | Batavo | Grego Morango | 100 | Gramas
3,IOGURTE PAULISTA POLPA MORANGO 600GR,Iogurte,,Paulista Morango Polpa,None,Gramas,Iogurte | Paulista Morango Polpa | 600 | Gramas
4,IOGURTE DANONE 900G DANONINHO MORANGO,Iogurte,Danone,Danoninho Morango,None,Gramas,Iogurte | Danone | Danoninho Morango | 900 | G...
5,36-IOGURTE INT MAMAO VIGOR 12X170G - 582825,Iogurte,Vigor,Int Mamão,None,Gramas,Iogurte | Vigor | Int Mamão | 170 | Gramas
6,30-IOGURTE INT MAMAO VIGOR 12X170G - 582825,Iogurte,Vigor,Int,None,Gramas,Iogurte | Vigor | Int | 170 | Gramas
7,35-IOGURTE INT MAMAO VIGOR 12X170G - 582825,Iogurte,Vigor,Int Mamão,None,Gramas,Iogurte | Vigor | Int Mamão | 170 | Gramas
8,10-IOGURTE INT MAMAO VIGOR 12X170G - 582825,Iogurte,Vigor,Int Mamão,None,Gramas,Iogurte | Vigor | Int Mamão | 170 | Gramas
9,13-IOGURTE INT MAMAO VIGOR 12X170G - 582825,Iogurte,Vigor,Int,None,Gramas,Iogurte | Vigor | Int | 170 | Gramas



C. Shampoo (variante léxica) | Query: SHAMPOO ELSEVE OLEO EXTRAORDINARIO NUTRICAO 400ML
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.


,XPROD,Categoria,Marca,Tipo_Variante,Qtd,UoM,Produto_Canonico
0,COND ELSEVE NUTRICAO 10X 400ML,Condicionador,Elseve,Nutrição,None,Mililitros,Condicionador | Elseve | Nutrição | 400 | Mili...
1,SH ELSEVE OLEO EXTRAORD NUTRICAO 1X400ML RH132...,Shampoo,Elseve,Óleo Extraordinário Nutrição,None,Mililitros,Shampoo | Elseve | Óleo Extraordinário Nutriçã...
2,OLEO EXTRAORDINARIO DISCIPLINANTE:100ml,Shampoo,,Óleo Extraordinário Disciplinante,None,Mililitros,Shampoo | Óleo Extraordinário Disciplinante | ...
3,SH ELSEVE 400ML COLORVIVE,Shampoo,Elseve,Colorvive,None,Mililitros,Shampoo | Elseve | Colorvive | 400 | Mililitros
4,SH ELSEVE SUPREME CONTROL 4D 400ML,Shampoo,Elseve,Supreme Control 4D,None,Mililitros,Shampoo | Elseve | Supreme Control 4D | 400 | ...
5,SH ELSEVE 400ML ARGININA RESIST X3,Shampoo,Elseve,Resist X3,None,Mililitros,Shampoo | Elseve | Resist X3 | 400 | Mililitros
6,REVITRAT NUTRICAO SHAMPOO - 230ML -L.201407068,Shampoo,Revitrat,Nutrição,None,Mililitros,Shampoo | Revitrat | Nutrição | 230 | Mililitros
7,OLEO CAP ELSEVE EXTRAORD RESS100ML,Óleo,Elseve,Ress,None,Mililitros,Óleo | Elseve | Ress | 100 | Mililitros
8,BC OLEO ARGAN MASC NUTRICAO 60G,Óleo,Bc,Máscara Argan Nutrição,None,Gramas,Óleo | Bc | Máscara Argan Nutrição | 60 | Gramas
9,SHAMPOO ELSEVE LISS INTENSE EXTEME 200ML,Shampoo,Elseve,Liss Intense Extreme,None,Unidade,Shampoo | Elseve | Liss Intense Extreme | 1 | ...



D. Desodorante sem variante | Query: DES 400ml REXO
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.


,XPROD,Categoria,Marca,Tipo_Variante,Qtd,UoM,Produto_Canonico
0,NEUTROG BODY INT 400ML,Desodorante,,Intenso,None,Mililitros,Desodorante | Intenso | 400 | Mililitros
1,COND.JEJ.BABY 400ML,Condicionador,,Jejum Baby,None,Mililitros,Condicionador | Jejum Baby | 400 | Mililitros
2,CHARMING SP ARGA 400ML,Condicionador,Charming,SP Argan,None,Mililitros,Condicionador | Charming | SP Argan | 400 | Mi...
3,NIVEA SOFT MILK 400ML,Hidratante,Nivea,Milk Soft,None,Mililitros,Hidratante | Nivea | Milk Soft | 400 | Mililitros
4,COND.PANTENE HIDRATACAO 400ML,Condicionador,Pantene,Hidratação,None,Mililitros,Condicionador | Pantene | Hidratação | 400 | M...
5,COND FRUCTIS NORMAL 400ML,Condicionador,Fructis,Normal,None,Mililitros,Condicionador | Fructis | Normal | 400 | Milil...
6,SPRAY CHARMING ARGAN 400ML,Spray,Charming,Argan,None,Mililitros,Spray | Charming | Argan | 400 | Mililitros
7,LOCAO CLAREADORA 2 - 400ML,Loção,,Clareadora,None,Mililitros,Loção | Clareadora | 400 | Mililitros
8,SH ELSEVE 400ML COLORVIVE,Shampoo,Elseve,Colorvive,None,Mililitros,Shampoo | Elseve | Colorvive | 400 | Mililitros
9,DES NIVEA ROLL FEM INV BEW 3X50ML 15% DES,Desodorante,Nivea,Roll Feminino Inv,None,Mililitros,Desodorante | Nivea | Roll Feminino Inv | 50 |...



E. Desodorante com variante | Query: ANTI TRANS DES REX JASMIN 400ml
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.


,XPROD,Categoria,Marca,Tipo_Variante,Qtd,UoM,Produto_Canonico
0,DES RO FEM REX ACTIV,Desodorante,Rexona,Feminino Activ,None,Unidade,Desodorante | Rexona | Feminino Activ | 1 | Un...
1,DES.REX.ROL.M.SPORTFAN 50ML,Desodorante,Rexona,Roll-On Sport Fan,None,Mililitros,Desodorante | Rexona | Roll-On Sport Fan | 50 ...
2,DES.REX.CREME M.CLIN.S.ST.48GR,Desodorante,Rexona,Creme,None,Unidade,Desodorante | Rexona | Creme | 1 | Unidade
3,DES.REX.AERO F.WOM.S-PERF.105G,Desodorante,Rexona,Aerosol Feminino,None,Gramas,Desodorante | Rexona | Aerosol Feminino | 105 ...
4,BOZZANO DES AERO ANTI DRACO 12X90G,Desodorante,Bozzano,Aerosol Anti-Transpirante,None,Gramas,Desodorante | Bozzano | Aerosol Anti-Transpira...
5,DES.VICHY ANTI-TR.24H AE 125,Desodorante,Vichy,Anti-Transpirante 24H,None,Mililitros,Desodorante | Vichy | Anti-Transpirante 24H | ...
6,ESM TRANS LUAR 1X6,Transparente,,NaN,None,Unidade,Transparente | 1 | Unidade
7,PEDRA INOVE JASMIN 96X35G,Pedra,Jasmin,Inove,None,Gramas,Pedra | Jasmin | Inove | 35 | Gramas
8,DESINFETANTE SUPERBOM JASMIN 6X2LT,Desinfetante,Jasmin,Superbom,None,Mililitros,Desinfetante | Jasmin | Superbom | 2000 | Mili...
9,ESM TRANS BL TOP BLANC 1X6,Transparente,,Top Blanc,None,Unidade,Transparente | Top Blanc | 1 | Unidade


In [8]:
import pandas as pd
from pathlib import Path
import random

INPUT_CSV = Path('../data/AmostraDescMerc.csv')
ATTACK_CSV = Path('attack_nfe.csv')  
NORMALIZED_CSV = Path('normalized_nfe.csv')

SAMPLE_SIZE = 30
SEED = 42

if not INPUT_CSV.exists():
    raise FileNotFoundError(f'CSV de entrada não encontrado: {INPUT_CSV.resolve()}')

if 'motor_v2' not in globals():
    raise RuntimeError('motor_v2 não encontrado. Execute a célula de inicialização do motor antes desta.')

df_in = pd.read_csv(INPUT_CSV, sep='\t')
if 'Text' not in df_in.columns:
    raise ValueError("Coluna 'Text' não encontrada no CSV de entrada.")

originais = df_in['Text'].dropna().astype(str).tolist()
if SAMPLE_SIZE is not None:
    if SAMPLE_SIZE <= 0:
        raise ValueError('SAMPLE_SIZE deve ser > 0 ou None.')
    sample_n = min(SAMPLE_SIZE, len(originais))
    originais = pd.Series(originais).sample(n=sample_n, random_state=SEED).tolist()

attack_rng = random.Random(SEED)
atacados = [
    apply_text_attack(texto, attack_rng, intensity=motor_v2.text_attack_intensity)
    for texto in originais
]

pd.DataFrame({
    'original': originais,
    'attacked': atacados,
}).to_csv(ATTACK_CSV, index=False, encoding='utf-8-sig')

predicoes = inferir_em_lotes_v2(motor_v2, atacados, tamanho_lote=15, pausa=1.0)
predicoes = [post_process(p) for p in predicoes]

def montar_normalized(produto: dict) -> str:
    categoria = (produto.get('Categoria') or '').strip()
    marca = (produto.get('Marca') or '').strip()
    variante = (produto.get('Tipo_Variante') or '').strip()
    qtd = produto.get('Quantidade_Unitaria')
    uom = (produto.get('Unidade_Medida') or '').strip()

    partes_nome = [p for p in [categoria, marca, variante] if p]
    nome_composto = ' '.join(partes_nome).strip()

    try:
        q = float(qtd)
        qtd_fmt = str(int(q)) if q.is_integer() else str(q)
    except (TypeError, ValueError):
        qtd_fmt = str(qtd).strip() if qtd is not None else ''

    sufixo = ' '.join([p for p in [qtd_fmt, uom] if p]).strip()
    return f'{nome_composto} - {sufixo}'.strip(' -')

n = min(len(originais), len(atacados), len(predicoes))
if n < len(originais):
    print(f'Aviso: tamanho divergente entre entrada e predição. Exportando {n} linhas alinhadas.')

df_norm = pd.DataFrame({
    'original': originais[:n],
    'attacked': atacados[:n],
    'normalized': [montar_normalized(p) for p in predicoes[:n]],
})

df_norm.to_csv(NORMALIZED_CSV, index=False, encoding='utf-8-sig')

print(f'attack.csv gerado em: {ATTACK_CSV.resolve()} | linhas: {len(originais)}')
print(f'normalized.csv gerado em: {NORMALIZED_CSV.resolve()} | linhas: {len(df_norm)}')
display(df_norm.head(10))

Lote 1 (itens 1–15)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 14.
Tentativa 2/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 14.
Tentativa 3/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.
Lote 2 (itens 16–30)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.
attack.csv gerado em: /Users/brennohdev/development/poc-text-normalization/notebooks/attack_nfe.csv | linhas: 30
normalized.csv gerado em: /Users/brennohdev/development/poc-text-normalization/notebooks/normalized_nfe.csv | linhas: 30


,original,attacked,normalized
0,TEMPERO COMPLETO/02 12X930G,TEMPERO 12X930G COMPLETO/02,Tempero Completo - 930 Gramas
1,vassoura esfregao pl,esfregao vassoura pl,Esfregão Vassoura Plástico - 1 Unidade
2,PERU TEMP PERDIGAO KG KG / 1,PERU PERU TEMP PERDIGAO KG KG / 1,Carne Perdigão Peru Temperado - 1000 Gramas
3,Creme P Pentear 280 Ml 3d Oleo Monoi Soft Hair UN,Creme Pentear P 280 Ml 3d Oleo Monoi Soft Hair UN,Creme Pentear P 3D Óleo Monoi - 280 Mililitros
4,ESPONJ. BANHO PORUS (33930)**,BANHO PORUS (33930)**,Banho Porus - 1 Unidade
5,AROMATIZANTE SACHET OBSESSION,AROMATIZANTE SACHET,Aromatizante - 1 Unidade
6,Folhas Prontas Corporal 20 Folhas Ouro Depil S...,Folhas Prontas Corporal 20 Ouro Folhas Depil S...,Depilação San Folhas Prontas Corporal Ouro - 1...
7,TRESEM CD HID PROF 200,TRESEM HID CD PROF 200,Shampoo Tresemmé Hidratante Profundo - 200 Mil...
8,CUIDE-SE BEM ATIVADOR DE CACHOS 100ml,CUIDE-SE ATIVADOR BEM DE CACHOS 100ml,Ativador de Cachos Cuidese Bem - 100 Mililitros
9,PRO - LAPIS CARBON N. 10 - MARINHO,PRO - - LAPIS CARBON N. 10 - MARINHO,Lápis Carbono N. 10 Marinho - 1 Unidade


In [9]:
import pandas as pd
from pathlib import Path

df_sefaz = pd.read_csv(Path('../data/AmostraDescMerc.csv'), sep='\t')
amostra_30 = df_sefaz['Text'].dropna().sample(n=30, random_state=7).tolist()

print("Rodando extração em 30 itens reais (2 lotes de 15)...\n")
motor_inspecao = ResilientFiscalLLMEngineV2(
    model_name="gpt-4o-mini",
    text_attack_enabled=True,
    text_attack_seed=42,
    text_attack_intensity=0.25,
)
predicoes = inferir_em_lotes_v2(motor_inspecao, amostra_30, tamanho_lote=15, pausa=1.0)

produtos = [post_process(p) for p in predicoes]

df_inspecao = pd.DataFrame([
    {
        "XPROD_Original"      : orig,
        "Categoria"           : p.get("Categoria"),
        "Marca"               : p.get("Marca"),
        "Tipo_Variante"       : p.get("Tipo_Variante"),
        "Qtd_Unitaria"        : p.get("Quantidade_Unitaria"),
        "Qtd_Comercial"       : p.get("Quantidade_Comercial"),
        "UoM"                 : p.get("Unidade_Medida"),
        "Produto_Canonico"    : p.get("Produto_Canonico"),
    }
    for orig, p in zip(amostra_30, produtos)
])

pd.set_option('display.max_colwidth', 45)
display(df_inspecao)

Rodando extração em 30 itens reais (2 lotes de 15)...

Text attack config: {'enabled': True, 'seed': 42, 'intensity': 0.25}
Lote 1 (itens 1–15)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.
Lote 2 (itens 16–30)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Tentativa 2/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Tentativa 3/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Falha crítica: limite de retentativas atingido.


,XPROD_Original,Categoria,Marca,Tipo_Variante,Qtd_Unitaria,Qtd_Comercial,UoM,Produto_Canonico
0,- REF.: (82441429) - AGUA DE COLONIA 30ML...,Água de Colônia,Bb,Signature Man,30,1,Mililitros,Água De Colônia | Bb | Signature Man | 30...
1,MATT PULGÃO PRONTO USO - FR 500ML,Inseticida,Matt,Pronto Uso,500,1,Mililitros,Inseticida | Matt | Pronto Uso | 500 | Mi...
2,TINT KOLESTON 40 CAST MEDIO,Tintura,Koleston,40 Castanho Médio,1,1,Unidade,Tintura | Koleston | 40 Castanho Médio | ...
3,Perfume p/ Ambientes Acquaroma 200ml Prov...,Perfume de Ambiente,Acquaroma,Provence,200,1,Mililitros,Perfume De Ambiente | Acquaroma | Provenc...
4,AGUA SANIT VALOR 6X2LTS R014,Desinfetante,Valor,NaN,2000,6,Mililitros,Desinfetante | Valor | 2000 | Mililitros
5,"Perfume p/ Ambientes Acquaroma 1,15L Cedro",Perfume de Ambiente,Acquaroma,NaN,1150,1,Mililitros,Perfume De Ambiente | Acquaroma | 1150 | ...
6,613-FLANELA LARANJA - 30X50- 12X1 - CASA ...,Flanela,,Laranja 30x50,1,12,Unidade,Flanela | Laranja 30x50 | 1 | Unidade
7,SNACK MULTICEREAL INSPIRE TOMATE/ERVAS FI...,Snack,Inspire Finas,Multicereal Tomate/Ervas,1,12,Unidade,Snack | Inspire Finas | Multicereal Tomat...
8,ESM A H SB MET MUSA DE LIMAO 6X1 09 ML,Esmalte,Musa,Limão,9,6,Mililitros,Esmalte | Musa | Limão | 9 | Mililitros
9,CHOC LAKA OREO 150G,Chocolate,Oreo,Laka,150,1,Gramas,Chocolate | Oreo | Laka | 150 | Gramas


## 7. Benchmark Semântico

Duas métricas complementares, mais adequadas para normalização semântica do que F1 clássico:

**Por que não F1 de string matching?**
O F1 clássico penaliza normalizações corretas — `"OLEO EXTRAORD NUTRICAO"` vs `"Óleo Extraordinário Nutrição"` conta como erro mesmo sendo semanticamente idêntico. Para um problema de normalização, isso é uma métrica enganosa.

**Métricas adotadas:**

**7.1 Pairwise Consistency** — dado um conjunto de XPRODs que representam o mesmo SKU, mede a taxa em que o modelo gera o mesmo `Produto_Canonico` para todos. Avalia diretamente o objetivo do projeto: agrupar corretamente.

**7.2 LLM-as-a-Judge** — GPT-4o avalia semanticamente cada extração do gpt-4o-mini, com veredicto CORRETO / PARCIAL / INCORRETO e justificativa. Baseado em Zheng et al. (2023) — MT-Bench.

In [10]:
# ══════════════════════════════════════════════════════
# 8.1 PAIRWISE CONSISTENCY TEST
# Grupos de XPRODs que sabemos ser o mesmo SKU.
# Métrica: % de grupos onde todos os itens geram o mesmo Produto_Canonico.
# ══════════════════════════════════════════════════════

PAIRWISE_GROUPS = {
    "Iogurte Vigor Mamão 170g": [
        "36-IOGURTE INT MAMAO VIGOR 12X170G - 582825",
        "30-IOGURTE INT MAMAO VIGOR 12X170G - 582825",
        "10-IOGURTE INT MAMAO VIGOR 12X170G - 582825",
    ],
    "Shampoo Elseve Óleo Extraordinário 400ml": [
        "SH ELSEVE OLEO EXTRAORD NUTRICAO 1X400ML RH1325600",
        "SHAMPOO ELSEVE OLEO EXTRAORD 400ML",
        "SH ELSEVE OL EXTRAORD NUTRICAO 400ML",
    ],
    "Amaciante Baby Soft Carinho Azul 2L": [
        "AMAC BABY SOFT T. CARINHO AZUL 2 LT CX/0006",
        "AMACIANTE BABY SOFT CARINHO AZUL 2L",
        "AMAC BABY SOFT CARINHO AZUL 2LT",
    ],
    "Desodorante Rexona Jasmim 400ml": [
        "ANTI TRANS DES REX JASMIN 400ml",
        "DES REXONA JASMIN 400ML",
        "DESODORANTE REX JASMIM 400 ML",
    ],
    "Detergente Líquido Minuano Limão 500ml": [
        "DETERG LIQ MINUANO LIMAO 500ML CX/0024/UN",
        "DETERGENTE LIQUIDO MINUANO LIMAO 500ML",
        "DET LIQ MINUANO LIMAO 500 ML",
    ],
}

def run_pairwise_consistency(engine, groups: dict) -> dict:
    """Para cada grupo, extrai Produto_Canonico de todos os itens e verifica consistência."""
    resultados = {}
    todos_itens = [item for items in groups.values() for item in items]

    print(f"Enviando {len(todos_itens)} itens para teste de consistência...")
    predicoes = inferir_em_lotes_v2(engine, todos_itens, tamanho_lote=15, pausa=1.0)
    predicoes_pp = [post_process(p) for p in predicoes]

    idx = 0
    grupos_ok = 0
    for grupo, itens in groups.items():
        canonicos = []
        for item in itens:
            pc = predicoes_pp[idx].get("Produto_Canonico", "")
            canonicos.append(pc)
            idx += 1
        consistente = len(set(canonicos)) == 1
        if consistente:
            grupos_ok += 1
        resultados[grupo] = {"itens": itens, "canonicos": canonicos, "consistente": consistente}

    taxa = grupos_ok / len(groups)
    print(f"\n{'='*60}")
    print(f"PAIRWISE CONSISTENCY: {grupos_ok}/{len(groups)} grupos consistentes ({taxa:.0%})")
    print(f"{'='*60}")
    for grupo, res in resultados.items():
        status = "✓" if res["consistente"] else "✗"
        print(f"\n{status} {grupo}")
        for item, canonico in zip(res["itens"], res["canonicos"]):
            print(f"   XPROD:    {item}")
            print(f"   Canônico: {canonico}")

    return resultados, taxa

motor_consistency = ResilientFiscalLLMEngineV2(
    model_name="gpt-4o-mini",
    text_attack_enabled=True,
    text_attack_seed=42,
    text_attack_intensity=0.25,
)
resultados_pairwise, taxa_consistency = run_pairwise_consistency(motor_consistency, PAIRWISE_GROUPS)

Text attack config: {'enabled': True, 'seed': 42, 'intensity': 0.25}
Enviando 15 itens para teste de consistência...
Lote 1 (itens 1–15)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.

PAIRWISE CONSISTENCY: 3/5 grupos consistentes (60%)

✓ Iogurte Vigor Mamão 170g
   XPROD:    36-IOGURTE INT MAMAO VIGOR 12X170G - 582825
   Canônico: Iogurte | Vigor | Mamão | 170 | Gramas
   XPROD:    30-IOGURTE INT MAMAO VIGOR 12X170G - 582825
   Canônico: Iogurte | Vigor | Mamão | 170 | Gramas
   XPROD:    10-IOGURTE INT MAMAO VIGOR 12X170G - 582825
   Canônico: Iogurte | Vigor | Mamão | 170 | Gramas

✗ Shampoo Elseve Óleo Extraordinário 400ml
   XPROD:    SH ELSEVE OLEO EXTRAORD NUTRICAO 1X400ML RH1325600
   Canônico: Shampoo | Elseve | Óleo Extraordinário Nutrição | 400 | Mililitros
   XPROD:    SHAMPOO ELSEVE OLEO EXTRAORD 400ML
   Canônico: Shampoo | Elseve | Óleo Extraordinário | 400 | Mililitros
   XPROD:    SH ELSEVE OL EXTRAORD NUTRICAO 400ML
   Canônico:

### 8.2 LLM-as-a-Judge

O GPT-4o avalia cada extração semanticamente: dado o XPROD original e o
`Produto_Canonico` gerado, o juiz responde se a extração está correta,
parcialmente correta ou incorreta — com justificativa.

Isso elimina a necessidade de gabarito manual para avaliar campos semânticos
como Categoria, Marca e Tipo_Variante.

In [11]:
# ══════════════════════════════════════════════════════
# 8.2 LLM-AS-A-JUDGE
# GPT-4o avalia semanticamente cada extração.
# ══════════════════════════════════════════════════════

import pandas as pd

JUDGE_SYSTEM_PROMPT = """
Você é um avaliador especialista em dados fiscais brasileiros (NF-e).
Dado um XPROD original e uma extração estruturada, avalie se a extração está correta.

Responda APENAS com JSON no formato:
{
  "veredicto": "CORRETO" | "PARCIAL" | "INCORRETO",
  "justificativa": "string curta explicando o veredicto",
  "erros": ["lista de campos com problema, se houver"]
}

Critérios:
- CORRETO: todos os campos corretos semanticamente.
- PARCIAL: 1-2 campos com erro menor.
- INCORRETO: categoria errada, marca inventada, ou quantidade_unitaria errada.

Sobre quantidade:
- Quantidade_Unitaria = quantidade do produto individual (o item em si).
- Quantidade_Comercial = total comercializado na nota (fardo/caixa).
- "8X6X90G": Qtd_Unitaria=90g, Qtd_Comercial=48. Ambos devem estar corretos.
- "6X2LTS": Qtd_Unitaria=2L, Qtd_Comercial=6. Ambos devem estar corretos.

Seja tolerante com: acentuação, capitalização, sinônimos próximos.
Seja estrito com: quantidade_unitaria multiplicada por fardo, categoria trocada, marca inventada.
"""

def llm_judge(client, xprod: str, extracao: dict, model: str = "gpt-4o") -> dict:
    user_msg = f"""XPROD original: "{xprod}"

Extração gerada:
- Categoria: {extracao.get("Categoria")}
- Marca: {extracao.get("Marca")}
- Tipo_Variante: {extracao.get("Tipo_Variante")}
- Quantidade_Unitaria: {extracao.get("Quantidade_Unitaria")}
- Quantidade_Comercial: {extracao.get("Quantidade_Comercial")}
- Unidade_Medida: {extracao.get("Unidade_Medida")}
- Produto_Canonico: {extracao.get("Produto_Canonico")}

Avalie a extração."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)


def run_llm_judge_benchmark(xprods, predicoes, sample_size=30, judge_model="gpt-4o"):
    from openai import OpenAI
    client = OpenAI()
    amostra = list(zip(xprods, predicoes))[:sample_size]
    registros = []
    print(f"Avaliando {len(amostra)} itens com {judge_model} como juiz...\n")
    for i, (xprod, pred) in enumerate(amostra, 1):
        pred_pp = post_process(pred)
        try:
            veredicto = llm_judge(client, xprod, pred_pp, model=judge_model)
        except Exception as e:
            veredicto = {"veredicto": "ERRO", "justificativa": str(e), "erros": []}
        registros.append({
            "XPROD": xprod,
            "Produto_Canonico": pred_pp.get("Produto_Canonico"),
            "Qtd_Unitaria": pred_pp.get("Quantidade_Unitaria"),
            "Qtd_Comercial": pred_pp.get("Quantidade_Comercial"),
            "Veredicto": veredicto.get("veredicto"),
            "Justificativa": veredicto.get("justificativa"),
            "Erros": ", ".join(veredicto.get("erros", [])),
        })
        if i % 5 == 0:
            print(f"  {i}/{len(amostra)} avaliados...")
        time.sleep(0.5)

    df = pd.DataFrame(registros)
    print(f"\n{'='*60}")
    print(f"RELATÓRIO LLM-AS-A-JUDGE ({judge_model})")
    print(f"{'='*60}")
    contagem = df["Veredicto"].value_counts()
    total = len(df)
    for v, n in contagem.items():
        print(f"  {v:<12}: {n:>3} ({n/total:.0%})")
    score = (contagem.get("CORRETO",0) + 0.5*contagem.get("PARCIAL",0)) / total
    print(f"{'='*60}")
    print(f"  SCORE SEMÂNTICO: {score:.0%}  (corretos + 0.5×parciais)")
    print(f"{'='*60}\n")
    print("Casos INCORRETOS ou PARCIAIS:")
    display(df[df["Veredicto"].isin(["INCORRETO","PARCIAL"])][
        ["XPROD","Produto_Canonico","Qtd_Unitaria","Qtd_Comercial","Veredicto","Justificativa","Erros"]
    ])
    return df

from pathlib import Path
df_30 = pd.read_csv(Path('../data/AmostraDescMerc.csv'), sep='\t')
amostra_30_xprods = df_30['Text'].dropna().sample(n=30, random_state=7).tolist()

motor_judge = ResilientFiscalLLMEngineV2(
    model_name="gpt-4o-mini",
    text_attack_enabled=True,
    text_attack_seed=42,
    text_attack_intensity=0.25,
)
predicoes_30 = inferir_em_lotes_v2(motor_judge, amostra_30_xprods, tamanho_lote=15, pausa=1.0)

df_judge = run_llm_judge_benchmark(
    xprods=amostra_30_xprods,
    predicoes=predicoes_30,
    sample_size=30,
    judge_model="gpt-4o"
)

Text attack config: {'enabled': True, 'seed': 42, 'intensity': 0.25}
Lote 1 (itens 1–15)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.
Lote 2 (itens 16–30)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Tentativa 2/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Tentativa 3/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Falha crítica: limite de retentativas atingido.
Avaliando 15 itens com gpt-4o como juiz...

  5/15 avaliados...
  10/15 avaliados...
  15/15 avaliados...

RELATÓRIO LLM-AS-A-JUDGE (gpt-4o)
  CORRETO     :  10 (67%)
  INCORRETO   :   4 (27%)
  PARCIAL     :   1 (7%)
  SCORE SEMÂNTICO: 70%  (corretos + 0.5×parciais)

Casos INCORRETOS ou PARCIAIS:


,XPROD,Produto_Canonico,Qtd_Unitaria,Qtd_Comercial,Veredicto,Justificativa,Erros
4,AGUA SANIT VALOR 6X2LTS R014,Desinfetante | Valor | 2000 | Mililitros,2000,6,INCORRETO,"A categoria está errada, pois 'AGUA SANIT...","Categoria, Unidade_Medida"
7,SNACK MULTICEREAL INSPIRE TOMATE/ERVAS FI...,Snack | Inspire Finas | Multicereal Tomat...,1,12,INCORRETO,A quantidade_unitaria está errada e a mar...,"Quantidade_Unitaria, Marca"
9,CHOC LAKA OREO 150G,Chocolate | Oreo | Laka | 150 | Gramas,150,1,INCORRETO,"A marca está incorreta, pois 'Oreo' é uma...",Marca
11,SACO P/LIXO PROEZA 30LT 25X10X1 PRETO,Saco De Lixo | Proeza | Preto 25x10x1 | 1...,1,1,PARCIAL,"A extração está quase correta, mas a quan...",Quantidade_Comercial
12,Conjunto Pentes Romance Ref. Sm 170 Condo...,Conjunto De Pentes | Condor | Romance | 1...,1,170,INCORRETO,A quantidade comercial está incorreta e n...,Quantidade_Comercial


In [12]:
# Text Attack OFF vs ON (mesma amostra)

from pathlib import Path
import pandas as pd


def calcular_score_semantico(df_resultado: pd.DataFrame) -> float:
    contagem = df_resultado["Veredicto"].value_counts()
    total = len(df_resultado)
    if total == 0:
        return 0.0
    return (contagem.get("CORRETO", 0) + 0.5 * contagem.get("PARCIAL", 0)) / total


def rodar_ab_attack_vs_no_attack(sample_size: int = 30, seed: int = 7):
    df_base = pd.read_csv(Path('../data/AmostraDescMerc.csv'), sep='\t')
    amostra = df_base['Text'].dropna().sample(n=sample_size, random_state=seed).tolist()

    print(f"Amostra fixa: {sample_size} itens | seed={seed}")

    # A: sem ataque
    motor_off = ResilientFiscalLLMEngineV2(
        model_name="gpt-4o-mini",
        text_attack_enabled=False,
        text_attack_seed=42,
        text_attack_intensity=0.25,
    )
    pred_off = inferir_em_lotes_v2(motor_off, amostra, tamanho_lote=15, pausa=1.0)
    df_off = run_llm_judge_benchmark(
        xprods=amostra,
        predicoes=pred_off,
        sample_size=sample_size,
        judge_model="gpt-4o",
    )
    score_off = calcular_score_semantico(df_off)

    # B: com ataque
    motor_on = ResilientFiscalLLMEngineV2(
        model_name="gpt-4o-mini",
        text_attack_enabled=True,
        text_attack_seed=42,
        text_attack_intensity=0.25,
    )
    pred_on = inferir_em_lotes_v2(motor_on, amostra, tamanho_lote=15, pausa=1.0)
    df_on = run_llm_judge_benchmark(
        xprods=amostra,
        predicoes=pred_on,
        sample_size=sample_size,
        judge_model="gpt-4o",
    )
    score_on = calcular_score_semantico(df_on)

    delta = score_on - score_off

    print("\n" + "=" * 70)
    print("COMPARATIVO A/B — SCORE SEMÂNTICO")
    print("=" * 70)
    print(f"Attack OFF: {score_off:.0%}")
    print(f"Attack ON : {score_on:.0%}")
    print(f"Delta (ON - OFF): {delta:+.1%}")
    print("=" * 70)

    df_cmp = pd.DataFrame([
        {"Cenário": "Attack OFF", "Score_Semântico": score_off},
        {"Cenário": "Attack ON", "Score_Semântico": score_on},
        {"Cenário": "Delta (ON - OFF)", "Score_Semântico": delta},
    ])
    display(df_cmp)

    return {
        "score_off": score_off,
        "score_on": score_on,
        "delta": delta,
        "df_off": df_off,
        "df_on": df_on,
    }


ab_result = rodar_ab_attack_vs_no_attack(sample_size=30, seed=7)


Amostra fixa: 30 itens | seed=7
Text attack config: {'enabled': False, 'seed': 42, 'intensity': 0.25}
Lote 1 (itens 1–15)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.
Lote 2 (itens 16–30)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Tentativa 2/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Tentativa 3/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Falha crítica: limite de retentativas atingido.
Avaliando 15 itens com gpt-4o como juiz...

  5/15 avaliados...
  10/15 avaliados...
  15/15 avaliados...

RELATÓRIO LLM-AS-A-JUDGE (gpt-4o)
  CORRETO     :  11 (73%)
  PARCIAL     :   3 (20%)
  INCORRETO   :   1 (7%)
  SCORE SEMÂNTICO: 83%  (corretos + 0.5×parciais)

Casos INCORRETOS ou PARCIAIS:


,XPROD,Produto_Canonico,Qtd_Unitaria,Qtd_Comercial,Veredicto,Justificativa,Erros
1,MATT PULGÃO PRONTO USO - FR 500ML,Inseticida | Pronto Uso | 500 | Mililitros,500,1,PARCIAL,"A extração está quase correta, mas a marc...",Marca
10,LIMP AZULIM 500ML TIRA LIMO SPRAY,Limpeza | Azulim Tira Limão Spray | 500 |...,500,1,PARCIAL,A extração contém um erro menor no campo ...,Tipo_Variante
11,SACO P/LIXO PROEZA 30LT 25X10X1 PRETO,Saco De Lixo | Proeza | 30L | 1 | Unidade,1,10,INCORRETO,"A quantidade unitária está errada, pois d...","Quantidade_Unitaria, Quantidade_Comercial"
13,LIMP AZULIM POWER PORCELANATO 750ML LAVANDA,Limpeza | Azulim Power Porcelanato Lavand...,750,1,PARCIAL,"A extração está quase correta, mas a marc...",Marca


Text attack config: {'enabled': True, 'seed': 42, 'intensity': 0.25}
Lote 1 (itens 1–15)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Validação aprovada.
Lote 2 (itens 16–30)...
Tentativa 1/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Tentativa 2/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Tentativa 3/3...
Enviando lote de 15 itens para gpt-4o-mini...
Rejeitado: Completude falhou: enviado 15, recebido 13.
Falha crítica: limite de retentativas atingido.
Avaliando 15 itens com gpt-4o como juiz...

  5/15 avaliados...
  10/15 avaliados...
  15/15 avaliados...

RELATÓRIO LLM-AS-A-JUDGE (gpt-4o)
  CORRETO     :   8 (53%)
  INCORRETO   :   7 (47%)
  SCORE SEMÂNTICO: 53%  (corretos + 0.5×parciais)

Casos INCORRETOS ou PARCIAIS:


,XPROD,Produto_Canonico,Qtd_Unitaria,Qtd_Comercial,Veredicto,Justificativa,Erros
4,AGUA SANIT VALOR 6X2LTS R014,Desinfetante | Valor | 2000 | Mililitros,2000,6,INCORRETO,"A categoria está errada, pois 'Água Sanit...","Categoria, Unidade_Medida"
6,613-FLANELA LARANJA - 30X50- 12X1 - CASA ...,Saco De Lixo | Proeza | 25x10x1 Preto | 1...,1,1,INCORRETO,"A categoria, marca e tipo_variantes estão...","Categoria, Marca, Tipo_Variante, Produto_..."
7,SNACK MULTICEREAL INSPIRE TOMATE/ERVAS FI...,Snack | Inspire Finas | Multicereal Tomat...,1,12,INCORRETO,A quantidade_unitaria está errada e a mar...,"Quantidade_Unitaria, Marca"
9,CHOC LAKA OREO 150G,Chocolate | Oreo | Laka | 150 | Gramas,150,1,INCORRETO,"A marca está incorreta, pois 'Oreo' é uma...",Marca
11,SACO P/LIXO PROEZA 30LT 25X10X1 PRETO,Lápis | St | Sombra Azul Claro | 1 | Unidade,1,1,INCORRETO,"A categoria, marca, tipo_variante e produ...","Categoria, Marca, Tipo_Variante, Produto_..."
12,Conjunto Pentes Romance Ref. Sm 170 Condo...,Flanela | Laranja 30x50 | 1 | Unidade,1,12,INCORRETO,A categoria e o produto canônico estão in...,"Categoria, Produto_Canonico, Marca"
14,4530438 ST LAPIS SOMBRA AZUL CL,Conjunto De Pentes | Condor | Romance | 1...,1,170,INCORRETO,"A categoria, marca e tipo_variante estão ...","Categoria, Marca, Tipo_Variante, Produto_..."



COMPARATIVO A/B — SCORE SEMÂNTICO
Attack OFF: 83%
Attack ON : 53%
Delta (ON - OFF): -30.0%


,Cenário,Score_Semântico
0,Attack OFF,0.833333
1,Attack ON,0.533333
2,Delta (ON - OFF),-0.300000


## 9. Teste de Compatibilidade com o Benchmark Padronizado

Lê o `attack.csv` compartilhado entre as duplas (colunas `id`, `original`, `tipo_variacao`, `attacked`),
passa cada `attacked` pelo motor **sem** aplicar text attack interno (o ataque já veio pronto),
e gera `normalized.csv` com as colunas `original`, `attacked`, `normalized` esperadas pelo benchmark de constancy.

**O que medir:** para cada `original`, as 6 variações de `attacked` devem produzir o mesmo `normalized`.
Se produzirem, constancy = 1.0. Se divergirem, constancy < 1.0.

In [21]:
# ══════════════════════════════════════════════════════════════════
# 9. GERAÇÃO DO normalized.csv PARA O BENCHMARK PADRONIZADO
# Lê o attack.csv compartilhado e gera normalized.csv compatível.
# Inclui pré-normalização simbólica (camada 1) e montagem
# determinística do normalized (camada 2).
# ══════════════════════════════════════════════════════════════════

import re
import pandas as pd
from pathlib import Path
from collections import Counter

ATTACK_CSV_BENCHMARK    = Path('attack.csv')
NORMALIZED_CSV_BENCHMARK = Path('normalized.csv')

# None = todos os 100 originals. Use um número menor para testes rápidos.
SAMPLE_ORIGINALS = 10


# ── CAMADA 1: Pré-normalização simbólica ─────────────────────────
# Aplicada no texto ANTES de enviar ao LLM.
# Objetivo: reduzir variação na entrada para que o modelo receba
# formas mais uniformes, independente do tipo de ataque recebido.

_PRE_RULES = [
    # Frações textuais de litro → ml
    (re.compile(r'1/2\s*l(?:itro)?s?\b', re.I),  '500ml'),
    (re.compile(r'meio\s+litro',          re.I),  '500ml'),
    # Litros com decimal → ml  (ex: "0,75L" → "750ml", "1.5 litros" → "1500ml")
    (re.compile(r'(\d+(?:[,.]\d+)?)\s*l(?:itro[s]?)?\b', re.I),
        lambda m: f"{int(float(m.group(1).replace(',', '.')) * 1000)}ml"),
    # Quilos (ex: "12 kilos" → "12kg")
    (re.compile(r'(\d+(?:[,.]\d+)?)\s*k(?:ilo[s]?|g[s]?)?\b', re.I),
        lambda m: f"{m.group(1).replace(' ', '').replace(',', '.')}kg"),
    # Gramas (ex: "200 gramas" → "200g")
    (re.compile(r'(\d+(?:[,.]\d+)?)\s*(?:gramas?|gr)\b', re.I),
        lambda m: f"{m.group(1).replace(' ', '')}g"),
    # Voltagem (ex: "110 volts" → "110v", "bi-volt" → "bivolt")
    (re.compile(r'bi[\s-]?volt(?:s)?\b', re.I), 'bivolt'),
    (re.compile(r'(\d+)\s*volt?s?\b',    re.I), lambda m: f"{m.group(1)}v"),
    # Espaços duplos
    (re.compile(r'\s{2,}'), ' '),
]

def pre_normalize(text: str) -> str:
    """Normaliza variações de unidade no texto antes do LLM."""
    for pattern, replacement in _PRE_RULES:
        if callable(replacement):
            text = pattern.sub(replacement, text)
        else:
            text = pattern.sub(replacement, text)
    return text.strip()


# ── CAMADA 2: Normalização dos campos pós-LLM ────────────────────
# Após a extração, normaliza os valores dos campos antes de montar
# a string final. Garante que "750ml" e "0,75L" virem o mesmo
# normalized, pois ambos chegam aqui como Quantidade=750, Unidade=Mililitros.

_UOM_CANONICAL = {
    'Mililitros':   'ml',
    'Litros':       'L',
    'Gramas':       'g',
    'Quilogramas':  'kg',
    'Metros':       'm',
    'Centímetros':  'cm',
    'Unidade':      '',
}

def _fmt_quantidade(qtd, uom: str) -> str:
    """Formata quantidade + unidade de forma canônica e compacta."""
    try:
        q = float(str(qtd).replace(',', '.'))
        q_str = str(int(q)) if q == int(q) else str(round(q, 3))
    except (TypeError, ValueError):
        return ''
    unit = _UOM_CANONICAL.get(uom, uom or '')
    if not unit:          # Unidade sem sufixo — omite do normalized
        return ''
    return f'{q_str}{unit}'

def montar_normalized(produto: dict) -> str:
    """
    Monta o normalized de forma determinística a partir dos campos extraídos.
    Ordem fixa: Categoria Marca Tipo_Variante - quantidade+unidade
    Campos None/vazios são omitidos silenciosamente.
    """
    categoria = (produto.get('Categoria') or '').strip().title()
    marca     = (produto.get('Marca')     or '').strip()
    variante  = (produto.get('Tipo_Variante') or '').strip()
    uom       = (produto.get('Unidade_Medida') or '').strip()
    qtd_fmt   = _fmt_quantidade(produto.get('Quantidade_Unitaria'), uom)

    partes_nome = [p for p in [categoria, marca, variante] if p]
    nome = ' '.join(partes_nome)

    return f'{nome} - {qtd_fmt}'.strip(' -') if qtd_fmt else nome


# ── Carrega e filtra o attack.csv ────────────────────────────────

if not ATTACK_CSV_BENCHMARK.exists():
    raise FileNotFoundError(f'attack.csv não encontrado: {ATTACK_CSV_BENCHMARK.resolve()}')

df_attack = pd.read_csv(ATTACK_CSV_BENCHMARK, encoding='utf-8-sig')
df_attack.columns = [c.strip().lower() for c in df_attack.columns]

for col in ('original', 'attacked'):
    if col not in df_attack.columns:
        raise ValueError(f'Coluna ausente no attack.csv: {col!r}')

df_attack['original'] = df_attack['original'].astype(str).str.strip()
df_attack['attacked'] = df_attack['attacked'].astype(str).str.strip()

if SAMPLE_ORIGINALS is not None:
    sel = df_attack['original'].unique()[:SAMPLE_ORIGINALS]
    df_attack = df_attack[df_attack['original'].isin(sel)].copy()

print(f'Linhas a processar: {len(df_attack)} '
      f'({df_attack["original"].nunique()} originals × '
      f'{len(df_attack) // df_attack["original"].nunique()} variações)')

# Aplica pré-normalização simbólica em cada attacked
df_attack['attacked_pre'] = df_attack['attacked'].apply(pre_normalize)
alterados = (df_attack['attacked'] != df_attack['attacked_pre']).sum()
print(f'Pré-normalização alterou {alterados}/{len(df_attack)} entradas.')


# ── Motor: sem text attack interno (ataque já veio pronto) ───────

class GenericNormalizerEngine(ResilientFiscalLLMEngineV2):
    """Motor com prompt genérico para produtos de e-commerce de domínio aberto."""

    def _build_system_prompt(self) -> str:
        return """
Você é um algoritmo de normalização de títulos de produtos de e-commerce.
Recebe descrições que podem conter abreviações, erros de digitação ou
notações de unidade variadas. Extraia os atributos canônicos.

Retorne APENAS JSON, sem texto extra.

CAMPOS OBRIGATÓRIOS:
- Categoria: tipo genérico do produto em português. Nunca use a marca.
- Marca: nome da marca. null se ausente. Não invente marcas.
- Tipo_Variante: modelo, especificação, cor ou variante relevante. null se ausente.
- Quantidade_Unitaria: valor numérico da medida do produto individual.
  Converta SEMPRE para a unidade base: litros → mililitros, kg mantém kg.
  Use 1 se não houver medida.
- Quantidade_Comercial: unidades no pacote/kit. 1 se não houver.
- Unidade_Medida: use exatamente um destes valores:
  Mililitros | Litros | Gramas | Quilogramas | Metros | Centímetros | Unidade

REGRAS:
- Abreviações → forma canônica (ex: "Microsc." → "Microscópio").
- Erros de digitação → corrija silenciosamente.
- Unidades variadas → mesmo valor numérico canônico
  ("0,75L", "750ml", "750 mililitros" → Quantidade_Unitaria=750, Unidade_Medida=Mililitros).
- Prefixos de quantidade comercial (ex: "Pack X 13", "11 Packs") → Quantidade_Comercial,
  não alteram a Categoria.

OUTPUT SCHEMA:
{"produtos_normalizados": [
  {"Categoria": "string", "Marca": "string|null", "Tipo_Variante": "string|null",
   "Quantidade_Unitaria": number, "Quantidade_Comercial": number, "Unidade_Medida": "string"}
]}
"""

motor_benchmark = GenericNormalizerEngine(
    model_name='gpt-4o-mini',
    text_attack_enabled=False,
    text_attack_seed=42,
    text_attack_intensity=0.0,
)

# Usa o texto pré-normalizado como entrada do LLM
attacked_pre_list = df_attack['attacked_pre'].tolist()
# Usa o texto pré-normalizado como entrada do LLM
attacked_pre_list = df_attack['attacked_pre'].tolist()
predicoes = inferir_em_lotes_v2(motor_benchmark, attacked_pre_list, tamanho_lote=1, pausa=0.3)
predicoes = [post_process(p) for p in predicoes]

n = min(len(df_attack), len(predicoes))
if n < len(df_attack):
    print(f'Aviso: predições incompletas. Exportando {n} linhas alinhadas.')

df_norm = df_attack.iloc[:n][['original', 'attacked']].copy()
df_norm['normalized'] = [montar_normalized(p) for p in predicoes[:n]]

df_norm.to_csv(NORMALIZED_CSV_BENCHMARK, index=False, encoding='utf-8-sig')
print(f'normalized.csv gerado: {NORMALIZED_CSV_BENCHMARK.resolve()} | {len(df_norm)} linhas')


# ── Prévia de constancy ───────────────────────────────────────────

print('\n── Prévia de constancy por original ──')
resultados = []
for original, grupo in df_norm.groupby('original', sort=False):
    counter = Counter(grupo['normalized'].tolist())
    group_size = len(grupo)
    dominant_count = max(counter.values())
    constancy = dominant_count / group_size
    dominant_norm = min(v for v, c in counter.items() if c == dominant_count)
    resultados.append({
        'original':           original[:60],
        'group_size':         group_size,
        'constancy':          round(constancy, 2),
        'dominant_normalized': dominant_norm[:60],
        'n_distinct':         len(counter),
    })

df_constancy = pd.DataFrame(resultados).sort_values('constancy')
mean_c = df_constancy['constancy'].mean()
print(f'Mean constancy: {mean_c:.2%}')
display(df_constancy)


Linhas a processar: 60 (10 originals × 6 variações)
Pré-normalização alterou 0/60 entradas.
Text attack config: {'enabled': False, 'seed': 42, 'intensity': 0.0}
Lote 1 (itens 1–1)...
Tentativa 1/3...
Enviando lote de 1 itens para gpt-4o-mini...
Validação aprovada.
Lote 2 (itens 2–2)...
Tentativa 1/3...
Enviando lote de 1 itens para gpt-4o-mini...
Validação aprovada.
Lote 3 (itens 3–3)...
Tentativa 1/3...
Enviando lote de 1 itens para gpt-4o-mini...
Validação aprovada.
Lote 4 (itens 4–4)...
Tentativa 1/3...
Enviando lote de 1 itens para gpt-4o-mini...
Validação aprovada.
Lote 5 (itens 5–5)...
Tentativa 1/3...
Enviando lote de 1 itens para gpt-4o-mini...
Validação aprovada.
Lote 6 (itens 6–6)...
Tentativa 1/3...
Enviando lote de 1 itens para gpt-4o-mini...
Validação aprovada.
Lote 7 (itens 7–7)...
Tentativa 1/3...
Enviando lote de 1 itens para gpt-4o-mini...
Validação aprovada.
Lote 8 (itens 8–8)...
Tentativa 1/3...
Enviando lote de 1 itens para gpt-4o-mini...
Validação aprovada.
Lote 9 

,original,group_size,constancy,dominant_normalized,n_distinct
4,Disyuntor Diferencial Bipolar 25a (30ma),6,0.17,"Disjuntor Diferencial Bipolar 0,025kA (0,...",6
5,Switch Ubiquiti Unifi 8 - Puertos Giga A...,6,0.17,Switch Ubiquiti Unifi 8 - 8 Portas Giga A...,6
8,Bocallave Hexagonal Impacto 1/2 - 7/8,6,0.17,Bocal Hexagonal Impacto 1/2 - 7/8,6
0,Microscópio Biológico Binocular Meopta Pr...,6,0.33,Microscópio Meopta Biológico Binocular,5
1,Nissan Versa,6,0.33,Automóvel Nissan Versa,4
2,Llave Contacto Yamaha Fz 16 + Tapa Tanque...,6,0.50,Acessórios Para Motos Yamaha Fz 16,4
3,Mini Pci Wireless + Bluetooth Samsung Np3...,6,0.67,Acessórios De Computador Samsung Mini PCI...,3
6,Fim De Curso - P/ Portão Acabamento Alumi...,6,0.67,Acessórios Para Portão Acabamento Alumínio,2
9,Moto G5 Libre Huella Zona Oeste,6,0.67,Celular Motorola G5,3
7,"Toalha Picnic Pic Nic Xadrez Oxford 1,50x...",6,1.00,Toalha Xadrez,1
